# Client routebepaling via wegennet Heerlen
 
Deze notebook laadt:
- Het wegennet uit `heerlen_edge_table.csv`
- De cliënten uit `clients.csv`
 
Vervolgens:
1. Bouwt een graph van het wegennet (nodes = kruispunten, edges = wegsegmenten).
2. Koppelt elke cliënt aan het dichtstbijzijnde knooppunt.
3. Berekent de kortste route (reistijd) tussen **alle** 100 cliënten.
4. Geeft de resultaten weer (afstandsmatrix, statistieken) en toont alles op een folium-kaart.


# 1. Importeer benodigde libraries

In [6]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
from shapely.geometry import Point
import folium
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

# 2. Laad de gegevens

In [7]:
# Laad edge tabel (wegen)
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f"Aantal edges: {len(edges_df)}")
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

# Bouw graph
G = nx.Graph()
node_coords = {}  # node_id -> (lon, lat)

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    start, end = row['u'], row['v']
    weight = row['travel_time_min']  # reistijd in minuten
    G.add_edge(start, end, weight=weight, geometry=geom)
    
    # Bewaar coördinaten van start‑ en eindpunt
    lon1, lat1 = coords[0]
    lon2, lat2 = coords[-1]
    node_coords[start] = (lon1, lat1)
    node_coords[end]   = (lon2, lat2)

print(f"Graph: {G.number_of_nodes()} knopen, {G.number_of_edges()} randen.")

# Maak array van node‑coördinaten voor k‑d‑tree
node_ids = list(node_coords.keys())
node_lons = np.array([node_coords[n][0] for n in node_ids])
node_lats = np.array([node_coords[n][1] for n in node_ids])
node_positions = np.column_stack((node_lons, node_lats))
tree = cKDTree(node_positions)

Aantal edges: 6824
Graph: 2926 knopen, 4055 randen.


# 3. Bouw het netwerk (graph)
We gebruiken `networkx` om een gewogen undirected graph te maken.
Elk segment heeft een gewicht = `travel_time_min` (reistijd in minuten).

In [8]:
# Laad clients
clients_df = pd.read_csv('../output/clients.csv')
print("Oorspronkelijke kolommen:", clients_df.columns.tolist())
print("Eerste 3 rijen:\n", clients_df.head(3))

# Detecteer de kolom die waarschijnlijk coördinaten bevat
coord_col = None
for col in clients_df.columns:
    # Kijk of de kolom strings bevat met een komma, spatie of puntkomma
    sample = clients_df[col].dropna().astype(str).iloc[0]
    if ',' in sample or ' ' in sample or ';' in sample:
        # Probeer of het te splitsen is in twee getallen
        parts = sample.replace(',', ' ').replace(';', ' ').split()
        if len(parts) == 2:
            try:
                float(parts[0]), float(parts[1])
                coord_col = col
                break
            except:
                pass

if coord_col is None:
    # Als niets gevonden, neem dan de eerste kolom
    coord_col = clients_df.columns[0]
    print(f"Geen duidelijke coördinatenkolom gevonden, gebruik {coord_col} als kolom met coördinaten.")
else:
    print(f"Gebruik kolom '{coord_col}' voor coördinaten.")

# Splits de kolom
# Bepaal scheidingsteken (komma, spatie of puntkomma)
def split_coords(s):
    s = str(s).replace(';', ' ').replace(',', ' ')
    parts = s.split()
    if len(parts) == 2:
        return float(parts[0]), float(parts[1])
    else:
        return np.nan, np.nan

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(lambda x: pd.Series(split_coords(x)))

# Verwijder rijen waar splitsing niet gelukt is
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f"Na splitsing: {len(clients_df)} cliënten overgebleven.")

# Optioneel: verwijder de oorspronkelijke kolom
clients_df.drop(columns=[coord_col], inplace=True)

print("Kolommen na splitsing:", clients_df.columns.tolist())
clients_df.head()


Oorspronkelijke kolommen: ['name', 'address', 'coordinates', 'care_arrangement', 'preferences', 'time_window_start', 'time_window_end', 'care_hours', 'dogs', 'cats', 'smokes']
Eerste 3 rijen:
        name                              address           coordinates  \
0  Client 1  Kloosterkoolhof 26D, 6415XT Heerlen   50.8910272 5.987952   
1  Client 2   Frans Halsstraat 6, 6415TH Heerlen  50.9003376 5.9904336   
2  Client 3         Hertstraat 1, 6414CH Heerlen  50.9155146 5.9745851   

  care_arrangement preferences time_window_start time_window_end  care_hours  \
0         HBH Plus     morning             08:00           12:00         2.0   
1         HBH Plus   afternoon             12:00           18:00         2.0   
2   Wash & Ironing     morning             08:00           12:00         2.5   

   dogs  cats  smokes  
0     2     0   False  
1     1     1   False  
2     0     0   False  
Gebruik kolom 'coordinates' voor coördinaten.
Na splitsing: 100 cliënten overgebleven.
Kolomm

,name,address,care_arrangement,preferences,time_window_start,time_window_end,care_hours,dogs,cats,smokes,lat,lon
0,Client 1,"Kloosterkoolhof 26D, 6415XT Heerlen",HBH Plus,morning,08:00,12:00,2.0,2,0,False,50.891027,5.987952
1,Client 2,"Frans Halsstraat 6, 6415TH Heerlen",HBH Plus,afternoon,12:00,18:00,2.0,1,1,False,50.900338,5.990434
2,Client 3,"Hertstraat 1, 6414CH Heerlen",Wash & Ironing,morning,08:00,12:00,2.5,0,0,False,50.915515,5.974585
3,Client 4,"Paulus Potterstraat 5, 6415TW Heerlen",Wash & Ironing,morning,08:00,12:00,2.0,0,1,False,50.899287,5.992158
4,Client 5,"Heerenweg 205A, 6414AG Heerlen",Wash & Ironing,afternoon,12:00,18:00,2.0,0,0,True,50.922154,5.975525


# 4. Koppel elke cliënt aan het dichtstbijzijnde knooppunt

In [9]:
def nearest_node(lon, lat):
    dist, idx = tree.query([lon, lat])
    return node_ids[idx]

clients_df['node'] = clients_df.apply(lambda row: nearest_node(row['lon'], row['lat']), axis=1)

print("Aantal cliënten zonder knooppunt:", clients_df['node'].isna().sum())
unique_nodes = clients_df['node'].unique()
print(f"Aantal unieke knooppunten onder cliënten: {len(unique_nodes)}")

Aantal cliënten zonder knooppunt: 0
Aantal unieke knooppunten onder cliënten: 89


# 5. Bereken alle onderlinge kortste pad reistijden
We gebruiken `nx.all_pairs_dijkstra_path_length` om alle combinaties te berekenen.
Dit kan even duren, maar met 100 knooppunten is het prima.

In [10]:
# Verzamel alle unieke cliëntknooppunten
client_nodes = clients_df['node'].unique()
n_clients = len(client_nodes)

# Maak mapping node -> index
node_to_idx = {node: i for i, node in enumerate(client_nodes)}

# Initialiseer matrix
dist_matrix = np.full((n_clients, n_clients), np.inf)

# Bereken voor elke bronnode de afstanden naar alle andere knooppunten
for src_node in client_nodes:
    lengths = nx.single_source_dijkstra_path_length(G, src_node, weight='weight')
    src_idx = node_to_idx[src_node]
    for dst_node in client_nodes:
        if dst_node in lengths:
            dst_idx = node_to_idx[dst_node]
            dist_matrix[src_idx, dst_idx] = lengths[dst_node]

print("Afstandsmatrix berekend, vorm:", dist_matrix.shape)

# Controleer op onbereikbare paren
inf_count = np.sum(np.isinf(dist_matrix))
if inf_count > 0:
    print(f"Waarschuwing: {inf_count} koppels zijn niet verbonden (oneindig).")
else:
    print("Alle koppels zijn bereikbaar!")

# Sla matrix op
pd.DataFrame(dist_matrix, index=client_nodes, columns=client_nodes).to_csv('client_distance_matrix.csv')
print("Afstandsmatrix opgeslagen als client_distance_matrix.csv")

# %%
# Enkele statistieken
finite = dist_matrix[~np.isinf(dist_matrix)]
print("\nStatistieken van onderlinge reistijden (minuten):")
print(f"  Gemiddeld : {np.mean(finite):.2f}")
print(f"  Mediaan   : {np.median(finite):.2f}")
print(f"  Minimum   : {np.min(finite[finite>0]):.2f} (niet-diagonaal)")
print(f"  Maximum   : {np.max(finite):.2f}")


Afstandsmatrix berekend, vorm: (89, 89)
Alle koppels zijn bereikbaar!
Afstandsmatrix opgeslagen als client_distance_matrix.csv

Statistieken van onderlinge reistijden (minuten):
  Gemiddeld : 4.50
  Mediaan   : 4.23
  Minimum   : 0.04 (niet-diagonaal)
  Maximum   : 11.72


# 6. Visualiseer cliënten en wegennet op een folium-kaart

In [ ]:
# Bepaal centrum
all_lons = np.concatenate([node_lons, clients_df['lon'].values])
all_lats = np.concatenate([node_lats, clients_df['lat'].values])
center_lon = np.mean(all_lons)
center_lat = np.mean(all_lats)

m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

# Teken wegen (vereenvoudigd)
highway_colors = {
    'motorway': '#e74c3c', 'motorway_link': '#e74c3c',
    'trunk': '#e67e22', 'trunk_link': '#e67e22',
    'primary': '#f1c40f', 'primary_link': '#f1c40f',
    'secondary': '#2ecc71', 'secondary_link': '#2ecc71',
    'tertiary': '#3498db', 'tertiary_link': '#3498db',
    'residential': '#9b59b6', 'living_street': '#1abc9c',
    'unclassified': '#95a5a6', 'service': '#bdc3c7',
}
def get_color(highway):
    if pd.isna(highway): return '#cccccc'
    key = str(highway).split('|')[0].strip()
    return highway_colors.get(key, '#cccccc')

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    latlon = [(lat, lon) for lon, lat in coords]
    color = get_color(row.get('highway'))
    folium.PolyLine(locations=latlon, color=color, weight=2, opacity=0.6).add_to(m)

# Teken cliënten
for idx, row in clients_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=1.0,
        popup=f"Client {idx}<br>Node: {row['node']}"
    ).add_to(m)

# Legenda
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 8px 12px; border-radius: 5px;
            font-size: 12px; font-family: sans-serif; box-shadow: 0 0 5px rgba(0,0,0,0.2);">
    <b>Legenda</b><br>
    <span style="color:blue;">●</span> Client<br>
    <span style="background:#e74c3c;">&nbsp;&nbsp;&nbsp;</span> Motorway<br>
    <span style="background:#e67e22;">&nbsp;&nbsp;&nbsp;</span> Trunk<br>
    <span style="background:#f1c40f;">&nbsp;&nbsp;&nbsp;</span> Primary<br>
    <span style="background:#2ecc71;">&nbsp;&nbsp;&nbsp;</span> Secondary<br>
    <span style="background:#3498db;">&nbsp;&nbsp;&nbsp;</span> Tertiary<br>
    <span style="background:#9b59b6;">&nbsp;&nbsp;&nbsp;</span> Residential<br>
    <span style="background:#95a5a6;">&nbsp;&nbsp;&nbsp;</span> Overig
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m.save('../output/client_network_map.html')
print("Kaart opgeslagen als client_network_map.html")

Kaart opgeslagen als client_network_map.html


# 7. Enkele voorbeeldroutes tonen (optioneel)
We kunnen voor een willekeurig client‑paar de route tekenen.

In [ ]:
if n_clients >= 2:
    src = clients_df.iloc[0]
    dst = clients_df.iloc[1]
    src_node = src['node']
    dst_node = dst['node']
    
    try:
        path_nodes = nx.shortest_path(G, source=src_node, target=dst_node, weight='weight')
        
        # Teken route op een aparte kaart
        route_map = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')
        
        # Teken alle wegen (lichtgrijs)
        for _, row in edges_df.iterrows():
            geom = row['geometry']
            coords = list(geom.coords)
            latlon = [(lat, lon) for lon, lat in coords]
            folium.PolyLine(locations=latlon, color='lightgray', weight=1, opacity=0.5).add_to(route_map)
        
        # Teken het gevonden pad in rood
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            # Zoek de edge
            edge = None
            mask = ((edges_df['u'] == u) & (edges_df['v'] == v)) | ((edges_df['u'] == v) & (edges_df['v'] == u))
            if mask.any():
                row = edges_df[mask].iloc[0]
                geom = row['geometry']
                coords = list(geom.coords)
                latlon = [(lat, lon) for lon, lat in coords]
                folium.PolyLine(locations=latlon, color='red', weight=5, opacity=0.8).add_to(route_map)
        
        # Teken start‑ en eindpunt
        folium.Marker(location=[src['lat'], src['lon']], icon=folium.Icon(color='green', icon='play')).add_to(route_map)
        folium.Marker(location=[dst['lat'], dst['lon']], icon=folium.Icon(color='red', icon='stop')).add_to(route_map)
        
        route_map.save('../output/example_route.html')
        print("Voorbeeldroute opgeslagen als example_route.html")
    except nx.NetworkXNoPath:
        print("Geen pad gevonden tussen de twee gekozen cliënten.")
else:
    print("Minder dan 2 cliënten, kan geen voorbeeldroute tekenen.")

Voorbeeldroute opgeslagen als example_route.html
